| Mục | Chi tiết cụ thể |
|:---|:---|
| **Dữ liệu đầu vào** | `movies.dat` và `ratings.dat` |
| **Chia dữ liệu** | Random shuffle → 30% train, 20% validation, 50% hidden test |
| **User-based Collaborative Filtering** | - Tính cosine similarity giữa các user<br>- Dự đoán rating bằng cách weighted average các user tương tự<br>- Không lọc top-k users (lấy tất cả các user có rating) |
| **Item-based Collaborative Filtering** | - Tính cosine similarity giữa các item (movies)<br>- Dự đoán rating bằng weighted average các item tương tự<br>- Cũng không lọc top-k items (lấy tất cả phim user đã xem) |
| **SVD (Model-based CF)** | - Dùng thư viện `surprise` (`SVD` class)<br>- Training trên trainset<br>- Dự đoán và tính RMSE trên validation và hidden test<br>- Dùng tham số mặc định (`n_factors=100`, `n_epochs=20`, `lr_all=0.005`, `reg_all=0.02`) |
| **Đánh giá RMSE** | - CF (User-based và Item-based): dùng `sqrt(mean_squared_error)`<br>- SVD: dùng `surprise.accuracy.rmse` |


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split as surprise_train_test_split
from surprise.model_selection import GridSearchCV
from surprise import accuracy
from sklearn.metrics.pairwise import cosine_similarity
from math import sqrt
from sklearn.metrics import mean_squared_error



In [2]:
movies = pd.read_csv("/Users/chi.nguyenth/Documents/DoAn_63133022_NguyenThiHaChi/dataset/1m/movies.dat", sep="::", engine="python", 
                     names=["movieId", "title", "genres"], encoding="latin1")

# Đọc ratings.dat và đổi tên cột MovieID -> ItemID
ratings = pd.read_csv("/Users/chi.nguyenth/Documents/DoAn_63133022_NguyenThiHaChi/dataset/1m/ratings.dat", sep="::", engine="python", 
                      names=["userId", "movieId", "rating", "timestamp"])

In [3]:
print(ratings.head())
print(movies.head())

   userId  movieId  rating  timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291
   movieId                               title                        genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy


In [4]:
# Xóa cột timestamp nếu không cần thiết
ratings = ratings.drop(columns=["timestamp"])

# Chuyển về định dạng phù hợp cho thư viện Surprise
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)


In [5]:
# 2. Chia dữ liệu
ratings_shuffled = ratings.sample(frac=1, random_state=42).reset_index(drop=True)
n_total = len(ratings_shuffled)
n_train = int(0.3 * n_total)
n_valid = int(0.2 * n_total)

train_data = ratings_shuffled[:n_train]
valid_data = ratings_shuffled[n_train:n_train + n_valid]
hidden_test_data = ratings_shuffled[n_train + n_valid:]

In [6]:
# 3. User-Item matrix
user_item_matrix = train_data.pivot(index='userId', columns='movieId', values='rating')
user_item_matrix_filled = user_item_matrix.fillna(0)

# 4. Tính Similarity
user_similarity = cosine_similarity(user_item_matrix_filled)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

item_similarity = cosine_similarity(user_item_matrix_filled.T)
item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)


In [7]:
# 5. Hàm dự đoán

def predict_user_based(user_id, movie_id):
    if movie_id not in user_item_matrix.columns or user_id not in user_item_matrix.index:
        return np.nan

    sim_users = user_similarity_df[user_id]
    movie_ratings = user_item_matrix[movie_id]
    mask = movie_ratings.notna()

    if mask.sum() == 0:
        return np.nan

    sim_scores = sim_users[mask]
    ratings = movie_ratings[mask]

    if sim_scores.sum() == 0:
        return np.nan

    prediction = np.dot(sim_scores, ratings) / sim_scores.sum()
    return prediction

def predict_item_based(user_id, movie_id):
    if user_id not in user_item_matrix.index:
        return np.nan

    user_ratings = user_item_matrix.loc[user_id]
    mask = user_ratings.notna()

    if mask.sum() == 0:
        return np.nan

    similar_items = item_similarity_df.get(movie_id)
    if similar_items is None:
        return np.nan

    similar_items = similar_items[mask]

    if similar_items.sum() == 0:
        return np.nan

    ratings = user_ratings[mask]
    prediction = np.dot(similar_items, ratings) / similar_items.sum()
    return prediction

In [8]:
# 6. Train SVD
reader = Reader(rating_scale=(0.5, 5.0))
trainset = Dataset.load_from_df(train_data[['userId', 'movieId', 'rating']], reader).build_full_trainset()

svd_model = SVD()
svd_model.fit(trainset)

In [9]:
# 7. Đánh giá

def evaluate_cf(predict_function, data):
    y_true = []
    y_pred = []

    for _, row in data.iterrows():
        uid = row['userId']
        iid = row['movieId']
        true_rating = row['rating']
        pred_rating = predict_function(uid, iid)

        if np.isnan(pred_rating):
            continue
        
        y_true.append(true_rating)
        y_pred.append(pred_rating)

    rmse = sqrt(mean_squared_error(y_true, y_pred))
    return rmse

def evaluate_svd(model, data):
    testset = list(zip(data['userId'], data['movieId'], data['rating']))
    predictions = [model.predict(uid, iid, r_ui) for (uid, iid, r_ui) in testset]
    rmse = accuracy.rmse(predictions, verbose=False)
    return rmse


In [10]:
# 8. Chạy đánh giá

print("=== Đánh giá trên Validation Set ===")
rmse_user_valid = evaluate_cf(predict_user_based, valid_data)
rmse_item_valid = evaluate_cf(predict_item_based, valid_data)
rmse_svd_valid = evaluate_svd(svd_model, valid_data)

print(f"User-Based CF Validation RMSE: {rmse_user_valid:.4f}")
print(f"Item-Based CF Validation RMSE: {rmse_item_valid:.4f}")
print(f"SVD Validation RMSE: {rmse_svd_valid:.4f}")

print("\n=== Đánh giá trên Hidden Test Set ===")
rmse_user_test = evaluate_cf(predict_user_based, hidden_test_data)
rmse_item_test = evaluate_cf(predict_item_based, hidden_test_data)
rmse_svd_test = evaluate_svd(svd_model, hidden_test_data)

print(f"User-Based CF Test RMSE: {rmse_user_test:.4f}")
print(f"Item-Based CF Test RMSE: {rmse_item_test:.4f}")
print(f"SVD Test RMSE: {rmse_svd_test:.4f}")

=== Đánh giá trên Validation Set ===
User-Based CF Validation RMSE: 0.9776
Item-Based CF Validation RMSE: 1.0112
SVD Validation RMSE: 0.9238

=== Đánh giá trên Hidden Test Set ===
User-Based CF Test RMSE: 0.9790
Item-Based CF Test RMSE: 1.0123
SVD Test RMSE: 0.9251
